# Training
Before running, complete notebook 01 and place the normalized train manifest/features in Drive. Set the run name uniquely per teammate/experiment. Training resumes from the latest valid checkpoint unless `--fresh` is supplied.

In [ ]:
from pathlib import Path
import os,sys,subprocess
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
candidates=[Path.cwd(),Path('/content/HMNet_SETG'),Path('/content/drive/MyDrive/nlvms/HMNet_SETG')]
ROOT=next((p for p in candidates if (p/'README.md').exists()),None)
if ROOT is None: raise FileNotFoundError('Project folder not found; extract the ZIP first.')
os.chdir(ROOT); sys.path.insert(0,str(ROOT))
from src.utils.config import load_config
from omegaconf import OmegaConf
MANIFEST=os.environ.get('HMNET_TRAIN_MANIFEST','/content/drive/MyDrive/nlvms/features/qvhighlights/train.jsonl')
FEATURE_ROOT=os.environ.get('HMNET_FEATURE_ROOT','/content/drive/MyDrive/nlvms/features/qvhighlights')
RUN_NAME=os.environ.get('HMNET_RUN_NAME','qvh_hmnet_baseline')
runtime_cfg=load_config('configs/qvhighlights.yaml',overrides={'paths.drive_root':'/content/drive/MyDrive/nlvms'})
CONFIG=os.environ.get('HMNET_CONFIG','configs/_colab_runtime.yaml')
if 'HMNET_CONFIG' not in os.environ: OmegaConf.save(runtime_cfg,CONFIG)
assert Path(MANIFEST).exists(), f'Missing manifest: {MANIFEST}'
cmd=[sys.executable,'scripts/train.py','--config',CONFIG,'--manifest',MANIFEST,'--feature-root',FEATURE_ROOT,'--run-name',RUN_NAME]

In [ ]:
VAL_MANIFEST=os.environ.get('HMNET_VAL_MANIFEST')
if VAL_MANIFEST and Path(VAL_MANIFEST).exists():
    cmd += ['--val-manifest', VAL_MANIFEST]
print('Validation manifest:', VAL_MANIFEST)

In [ ]:
print('Running:', ' '.join(cmd))
subprocess.run(cmd,check=True)